### Biomethane Supply-Chain (Robust) Linear Optimization (Biogoals.TE-LP tool)

**Authors:** Davide Carecci, Benjamin Riedel  
**Created:** April 2024  
**Last Modified:** October 23, 2025  
**Version:** 1.1  
**Institution:** Politecnico di Milano  

---

#### Revision History

| Date       | Version | Author(s)              | Description                                                                       |
|------------|---------|------------------------|-----------------------------------------------------------------------------------|
| 2024-05-22 | 1.0     | D. Carecci, B. Riedel  | Initial implementation for PoliMi Robust Optimization course                      |
| 2025-10-23 | 1.1     | D. Carecci             | Code ando folder cleaning and documentation for handle over a copy to A2A S.p.A   |

---
<h5>
WARNING!!!
The following code exploits a free Mosek licence the course "061652 ROBUST OPTIMIZATION" offered at Politecnico di Milano (expiration April 25th 2024). If you have error messages informing you about licencing issues, you may try uncommenting the installation lines for Gurobi. Otherwise, we recommend that you obtain your own licence of either Mosek (https://www.gurobi.com/) or Gurobi (https://www.mosek.com/products/trial/).

### Preliminaries

In [ ]:
# RUN THIS CELL ONLY ONCE TO SET UP THE ENVIRONMENT (FIRST TIME YOU OPEN THE NOTEBOOK)
# ------------------------------------------------------------------------------------- #
# INSTALL THE MAIN PACKAGES NEEDED FOR THE PROGRAM TO RUN i.e. RSOME (AND MOSEK OR GUROBI SOLVERS)
# Use "!pip" when using Google Colab
# Use "%pip" when running this notebook locally (better to avoid import errors if different Python environments are present locally)

# %pip install rsome
# !pip install mosek
# !rm mosek.lic
# !git clone https://github.com/erickdelage/80624
# !cp ./80624/mosek.lic .
# !rm -r ./80624
# !mkdir -p /root/mosek
# !cp ./mosek.lic /root/mosek
# Uncomment the following line to install Gurobi instead of MOSEK
# NB https://pypi.gurobi.com is deprecated, use the Gurobi website to create an account and get the installation instructions
#!pip install -i https://pypi.gurobi.com gurobipy

# Alternative: more robust installation method to install directly from Jupyter notebooks
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "rsome"])

In [ ]:
# IMPORT THE MAIN PACKAGES NEEDED FOR THE PROGRAM TO RUN
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rsome as rso
from rsome import ro
# from rsome import msk_solver as my_solver  #Import Mosek solver interface
# from rsome import grb_solver as my_solver  #Import Gurobi solver interface
from my_utilities import *  #Import all custom utility functions
from bmp_model import *  #Import BMP model functions


### Load and prepare the input data


In [ ]:
# LOAD INPUT DATA FROM EXCEL FILE
# Declare the current date for the naming of the output files purposes
current_date = datetime.now().date() # Get the current date
# formatted_date = current_date.strftime('%d.%m.%Y') # Format the date as a string

# Load data from the Excel file using the custom utility function
file_path = 'C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Robust Optimization/Project/Project_database.xlsx'
data = read_excel_file(file_path)

In [ ]:
# DEFINE SOME PROBLEM CONSTANTS AND THE BOUNDS FOR THE OPTMIMIZATION PROBLEM
n = data['Item characteristics'].shape[0] # number of items (co-feedstocks)
m = data['Ct'].shape[0] # number of available suppliers 

Vr = 2*1350+3179 # size of the biomethaneplant i.e. volume (m^3)
N = 365 # purchase horizon (days). Set this coherently with the data of the 'Availability' sheet in the Excel file!!
p_ch4_mwh = 110 # (fixed) biomethane selling cost (€/MWh)
p_ch4 = p_ch4_mwh *0.01035 # (fixed) biomethane selling cost converted in (€/Nm^{3}_{CH4})

# Define the bounds for the optimization problem (in the math. formulation: k_{min}, k_{max} for each k \in {VS, TKN, C/N, TAN, TAC, LI, TVFA} and HRT_{min}, HRT_{max} (last row)) 
bnds = np.array([
        [0, 120], # Constraints on total input diet volatile solids (VS (g_{VS}/ton_{FM}))
        [0.2, 4], # Constraints on total Kjeldahl nitrogen (TKN (kg_{N}/ton_{FM}))
        [20, 40], # Constraints on the carbon over nitrogen ratio (C/N (mol_C/mol_N))
        [0, 4], # Constraints on total ammonia nitrogen (TAN (kg_{N}/ton_{FM}))
        [5, 15], # Constraints on total alkalinity (TAC (kg_{CaCO3}/ton_{FM}))
        [0, 10], # Constraints on lipid content (LI (kg_{LI}/ton_{FM}))
        [0, 10], # Constraints on total volatile fatty acids (TVFA (kg_{Hac}/ton_{FM})...unit in acetate equivalent)
        [20,50] # Constraints on hydraulic retention time (HRT (days))...[HRT_{min}, HRT_{max}]
       ])

# Define the limits of the regulating authority (from Table 1A, 1B, Second Harvest, Slurries)
RL = [0.7, 0.3, 0.2, 0.4] # [Table 1A min fraction, Table 1B max fraction, Second Harvest max fraction, Slurries min fraction]

In [ ]:
# EXTRACT DATA FROM READING AND COMPUTE SOME QUANTITIES
VS = data['Item characteristics']['VS'].values.astype('float64') # Volatile Solids content (g_{VS}/ton_{FM})
BMPinf = data['Item characteristics']['BMPinf'].values.astype('float64') # Biochemical Methane Potential (BMP (Nm^3_{CH4}/ton_{FM}))
k = (np.concatenate(([data['Item characteristics'].values[:,3]], data['Item characteristics'].values[:,6:12].T), axis=0).T).astype('float64') # other substrate characteristics needed to enforce technical constraints
Cp = data['Purchase costs'].values[5:5+m,1:1+n].astype('float64') # feedstock purchase costs (€/ton_{FM})
A = data['Availability'].values[5:5+m,1:1+n].astype('float64') # feedstock availability matrix for each supplier (ton_{FM} over the entire purchase horizon N), rows: suppliers, columns: feedstocks
itemtype = data['Item characteristics'].values[:,-1] # Legislative cluster (P: products, S: slurry, P2: 2nd-turn cultures, W: waste, B: by-products)

# COMPUTE THE MATRIX OF TRANSPORTATION COSTS Ct
# Sum fixed tariff to 'distance tariff' (1 eur/km) and divide by 20 tons (nominal size of transportation truck)
Fixed_tariff = data['Item characteristics']['Fixed_tariff'].values.astype('float64') # fixed tariff (€/ton_{FM} for each type of feedstock)
d = data['Ct']['d'].values.astype('float64').reshape(-1, 1) # distance (km) from each supplier to the plant
matrix = np.tile(Fixed_tariff, (m, 1))
Ct = (matrix + d)/20 # note: '20' is the nominal size of the transportation truck (tons) # rows: suppliers, columns: feedstocks

In [ ]:
# CREATE THE BMP TIME CURVES FOR ALL THE FEEDSTOCKS
# RE-DECLARE SOME PARAMETERS TO CREATE THE BMP TIME CURVES
k_hydr = data['Item characteristics'].values[:,13].astype('float64') # hydrolysis rate constants (fast part of the BMP test i.e. of the rapidly biodegradable fraction) (1/day)
k_hyds = data['Item characteristics'].values[:,14].astype('float64') # hydrolysis rate constants (slow part of the BMP test i.e. of the slowly biodegradable fraction) (1/day)
BMPinf_r = data['Item characteristics'].values[:,15].astype('float64') # cumulative BMP produced by the rapidly biodegradable fraction (Nm^3_{CH4}/ton_{FM})
BMPinf_s = data['Item characteristics'].values[:,16].astype('float64') # cumulative BMP produced by the slowly biodegradable fraction (Nm^3_{CH4}/ton_{FM})

bmp_curve = [] # list to store the BMP time curves (1 point for each day of the time horizon)
for x0, y0, k1, k2 in zip(BMPinf_r, BMPinf_s, k_hydr, k_hyds): # loop over all the feedstocks
    t, bmp_day = evaluate_model(x0, y0, k1, k2, 50, 50) # evaluate the BMP model (two-pool first order kinetics model) over 50 days with 50 time points
    bmp_curve.append(bmp_day)

    # Plot the time response
    # fig, ax = plt.subplots(figsize=(12, 6))
    # ax.plot(t, bmp_day, label='BMP(t)', marker='o')
    # plt.xlabel('Time')
    # plt.ylabel('BMP(t)')
    # plt.legend()
    # plt.grid(True)
    # plt.show()
bmp_curve = np.array(bmp_curve, dtype=np.float64) # convert the list to a NumPy matrix for easier manipulation (rows: feedstocks, columns: time points)
# print(bmp_curve)

### Deterministic problem

In [ ]:
# If 'my_solver' is not present in local memory yet, define it here
if 'my_solver' not in globals():
    my_solver = None  # None allows model.solve() to choose default solver (scipy.optimize for LP problems)

In [ ]:
# SOLVE THE DETERMINISTIC PROBLEM (with consequent HRT optimization)
# Note: the 'simplified' formulation without HRT effect i.e. considering the ultimate BMPinf as BMP was delated in this code version (21.10.2025)

# Find indices correspondent to a certain cluster to apply the 'regulating authority constraints' (RL vector in the math. formulation)
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])

#Extract value from 'bmp_curve' at the given HRT
HRT = 20 # USER MODIFY THIS VALUE TO RANGE FOR DIFFERENT HRT's: must be an integer between HRT_{min} and HRT_{max} defined in 'bnds' 
HRTminus1 = 1/HRT
Nminus1 = 1/N
BMP = []
for i in range(len(data['Item characteristics'].values[:,0])):
    BMPsubstrate = bmp_curve[i][HRT-1]
    BMP.append(BMPsubstrate)
BMP = np.array(BMP)

# Create model
model=ro.Model('Biomethane supply-chain real')

# Define decision variables
x = model.dvar((n,m)) # The amounts of each feedstock to be purchased from each supplier over the purchase horizon N (ton_{FM})

# Objective function
model.max(Nminus1*(p_ch4*(VS*BMP/1000)@x.sum(axis=1)-(Cp.T*x).sum()-(Ct.T*x).sum())) # Maximize the net revenue over the purchase horizon N (€/day)

# Constraints
model.st(x[table1A].sum() >= RL[0]*x.sum()) # Constraints from regulating authority
model.st(x[table1B].sum() <= RL[1]*x.sum()) # Constraints from regulating authority
model.st(x[secondharvest].sum() <= RL[2]*x.sum()) # Constraints from regulating authority
model.st(x[slurries].sum() >= RL[3]*x.sum()) # Constraints from regulating authority
model.st(x <= A.T) # Availability constraint
for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st((BMPinf-BMP)@(x.sum(axis=1)) <= (0.15*BMPinf)@(x.sum(axis=1))) # Constraint to limit output residual BMP_{res} in digestate (environmental regulating authority limit)
model.st(Vr*N == HRT*x.sum()) # fixed HRT hard constraint
model.st(x.T[-1] == A[-1]) # Constraint to consume feedstocks produced at 'myplant' i.e. named 'z' in the input data Excel example file
model.st(x >= 0) # Non-negativity constraint

#Solve the model
model.solve(my_solver)
opt_obj_real = model.get() # Optimized value of the objective function
opt_x_real = x.get() # Optimized decision variables

# Print results
np.set_printoptions(suppress=True)
print('The net revenue for the purchase horizon of interest is', round(opt_obj_real,2),
      '€/day and the optimal fluxes of items to be purchased are \n', np.round(opt_x_real, decimals=2))

In [ ]:
# Compute some quantities of interest with the solution obtained above
x = opt_x_real # Optimal decision variables
J = opt_obj_real # Optimal objective function value
HRT = Vr*N/x.sum() # Check HRT value is correct
print(f'Checked HRT is {round(HRT,2)} days \n')
f_obj = p_ch4*(VS*BMP/1000)@x.sum(axis=1)/N # Biomethane gross revenue (€/day)
f_obj2 = (Cp.T*x).sum()/N # Feedstock's purchase cost (€/day)
f_obj3 = (Ct.T*x).sum()/N # Transportation cost (€/day)
print(f'Biomethane gross revenue is {round(f_obj,2)} €/day')
print(f'Feedstocks\' purchase cost is {round(f_obj2,2)} €/day')
print(f'Transportation cost is {round(f_obj3,2)} €/day')
print(f'Net revenue is {round(f_obj-f_obj2-f_obj3,2)} €/day \n')

# Optimal diet mix composition
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the input diet mix')
    diet.append(round(itempercentage,2))
print('')

# Save to Excel
results = [HRT, f_obj, f_obj2, f_obj3, J, diet]
file_path = 'Results.xlsx'
sheet_name = 'D-LP_real'
save_list_to_excel(results, file_path, sheet_name)

# Check that the regulating authority and technical constraints have been respected
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}') # Must be negative or zero
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}') # Must be positive or zero
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}') # Must be negative or zero
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}') # Must be positive or zero

### Design of uncertainty sets

In [ ]:
# DESIGN OF UNCERTAINTY SETS FOR THE ROBUST OPTIMIZATION PROBLEM
# In this example case, we did not have enough data historical realizations to design the uncertainty sets on those
# For this reason, we assumed a piori some distributions within min and max values given in the input data Excel file.
# If you have the exact realizations, you can directly use them to design the uncertainty sets 
# See user manual, "RobustOptimization_Project_Report.pdf" and lecture notes for further details.

# LOAD MEAN, MAXIMUM AND MINIMUM VALUES STORES IN THE INPUT DATA EXCEL FILE
# Of costs
cost_data = data['Purchase costs']
barley = cost_data['Barley silage'].values.astype('float64')[0:3]
chicken = cost_data['Chicken dung'].values.astype('float64')[0:3]
cow_d = cost_data['Cow dung'].values.astype('float64')[0:3] # Note "_d" stands for "dung"
cow_s = cost_data['Cow slurry'].values.astype('float64')[0:3] # Note "_s" stands for "slurry"
maize= cost_data['Maize silage'].values.astype('float64')[0:3]
pig = cost_data['Pig slurry'].values.astype('float64')[0:3]
sorghum = cost_data['Sorghum silage'].values.astype('float64')[0:3]
tomato = cost_data['Tomato peels'].values.astype('float64')[0:3]
triticale= cost_data['Triticale silage'].values.astype('float64')[0:3]
wheat = cost_data['Wheat byproducts'].values.astype('float64')[0:3]
feedstocks_costs = [barley,chicken,cow_d,cow_s,maize,pig,sorghum,tomato,triticale,wheat]
# print(barley)
# print(chicken)
# print(cow_d)
# print(cow_s)
# print(maize)
# print(pig)
# print(sorghum)
# print(tomato)
# print(triticale)
# print(wheat)

# Of availabilities #Note "_a" stands for "availability"
availability_data = data['Availability']
barley_a = availability_data['Barley silage'].values.astype('float64')[0:3]
chicken_a = availability_data['Chicken dung'].values.astype('float64')[0:3]
cow_d_a = availability_data['Cow dung'].values.astype('float64')[0:3]
cow_s_a = availability_data['Cow slurry'].values.astype('float64')[0:3]
maize_a= availability_data['Maize silage'].values.astype('float64')[0:3]
pig_a = availability_data['Pig slurry'].values.astype('float64')[0:3]
sorghum_a = availability_data['Sorghum silage'].values.astype('float64')[0:3]
tomato_a = availability_data['Tomato peels'].values.astype('float64')[0:3]
triticale_a = availability_data['Triticale silage'].values.astype('float64')[0:3]
wheat_a = availability_data['Wheat byproducts'].values.astype('float64')[0:3]
feedstocks_a = [barley_a,chicken_a,cow_d_a,cow_s_a,maize_a,pig_a,sorghum_a,tomato_a,triticale_a,wheat_a]
# print(barley_a)
# print(chicken_a)
# print(cow_d_a)
# print(cow_s_a)
# print(maize_a)
# print(pig_a)
# print(sorghum_a)
# print(tomato_a)
# print(triticale_a)
# print(wheat_a)

# Of BMP
bmp_data = [BMPinf,data['Item characteristics']['BMPinf Max'].values.astype('float64'),data['Item characteristics']['BMPinf Min'].values.astype('float64')]
# print(bmp_data)

In [ ]:
# GENERATION OF TIME SERIES DATA FOR COSTS AND AVAILABILITIES
# Data generation of costs
years = 5 # Number of years you want to simulate the availability of data
data_per_year = 52 # 52 realization/year (one/week)
np.random.seed(5) # Set random seed for reproducibility
feedstocks_costs_over_time=np.zeros([n,data_per_year*years])
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(barley,data_per_year,False)
  feedstocks_costs_over_time[0,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(chicken,data_per_year,False)
  feedstocks_costs_over_time[1,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(cow_d,data_per_year,False)
  feedstocks_costs_over_time[2,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(cow_s,data_per_year,False)
  feedstocks_costs_over_time[3,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(maize,data_per_year,False)
  feedstocks_costs_over_time[4,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(pig,data_per_year,False)
  feedstocks_costs_over_time[5,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(sorghum,data_per_year,False)
  feedstocks_costs_over_time[6,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(tomato,data_per_year,False)
  feedstocks_costs_over_time[7,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(triticale,data_per_year,False)
  feedstocks_costs_over_time[8,j:j+data_per_year]=d
for j in [0,data_per_year,data_per_year*2,data_per_year*3,data_per_year*4]:
  d=generate_cost_series(wheat,data_per_year,False)
  feedstocks_costs_over_time[9,j:j+data_per_year]=d
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[0])+1),feedstocks_costs_over_time[0],label='barley silage')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[1])+1),feedstocks_costs_over_time[1],label='chicken dung')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[2])+1),feedstocks_costs_over_time[2],label='cow dung')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[3])+1),feedstocks_costs_over_time[3],label='cow slurry')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[4])+1),feedstocks_costs_over_time[4],label='maize silage')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[5])+1),feedstocks_costs_over_time[5],label='pig slurry')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[6])+1),feedstocks_costs_over_time[6],label='sorghum silage')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[7])+1),feedstocks_costs_over_time[7],label='tomato peels')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[8])+1),feedstocks_costs_over_time[8],label='triticale silage')
# plt.plot(np.arange(1,len(feedstocks_costs_over_time[9])+1),feedstocks_costs_over_time[9],label='wheat')
# plt.legend(bbox_to_anchor=(1, 1), loc='upper left')
# plt.xlabel('Time (week)')
# plt.ylabel('Purchase cost (€ ton$_{FM}^{-1}$)')
# plt.grid()
# plt.xlim(0,260)
# Save in the images sub-folder placed in the current directory
# plt.savefig(f'images/Costdata_{current_date}.png', bbox_inches='tight')

# Data generation of availability
# One realization for each year
feedstocks_availability_over_time=np.zeros([n,years])
row=0
for i in feedstocks_a:
  for j in range(years):
    feedstocks_availability_over_time[row,j] = beta_sampler(i)
  row+=1

In [ ]:
# DESIGN OF THE UNCERTAINTY SETS (computation of the \Gamma dimension)
# Probability to meet the Chance constraint
Prob = 0.95 # Define the desired probability to meet chance constraint i.e. risk/safety level you want to assume against uncertainty. 90-95% is a common choice in practice.
eps = 1-Prob

# Compute Gamma_cost (data-driven approach)
mu_cost=np.mean(feedstocks_costs_over_time, axis=1)
tmp = feedstocks_costs_over_time - mu_cost.reshape(-1,1)@np.ones((1,feedstocks_costs_over_time.shape[1]))
tmp_max = abs(tmp).max(1)
Zs_cost = np.diag(1/tmp_max)@tmp
P_cost = np.diag(tmp_max)
tmp=np.sort(np.linalg.norm(Zs_cost,1,axis=0))
Gamma_cost=tmp[int(np.ceil((1-eps)*len(tmp)))-1]
print('Computed the uncertainty set dimension: Gamma_cost={0:0.6f}'.format(Gamma_cost))

# Compute Gamma_availability (data-driven approach)
mu_avail=np.mean(feedstocks_availability_over_time, axis=1)
tmp = feedstocks_availability_over_time - mu_avail.reshape(-1,1)@np.ones((1,feedstocks_availability_over_time.shape[1]))
tmp_max = abs(tmp).max(1)
Zs_avail = np.diag(1/tmp_max)@tmp
P_avail = np.diag(tmp_max)
tmp=np.sort(np.linalg.norm(Zs_avail,1,axis=0))
Gamma_availability=tmp[int(np.ceil((1-eps)*len(tmp)))-1]
print('Computed the uncertainty set dimension: Gamma_availability={0:0.6f}'.format(Gamma_availability))

# Compute Gamma_BMP (based on theoretical result for symmetric [-1,1] distributions)
Gamma_BMP = (2*n*np.log(1/eps))**0.5
print('Computed the uncertainty set dimension: Gamma_BMP={0:0.6f}'.format(Gamma_BMP))

In [ ]:
# Check constraint formulation for availability by taking one realization from Zs
# The random realization of costs is the index
cost_random_index = np.random.randint(0, feedstocks_costs_over_time.shape[1]-1)
# The random realization of availability is the index
availability_random_index = np.random.randint(0, feedstocks_availability_over_time.shape[1]-1)

deterministic_mean_a = [arr[0] for arr in feedstocks_a]
suppliersizes = []
for i in range(n):
    column_a = A[:,i]/deterministic_mean_a[i]
    suppliersizes.append(column_a)
suppliersizes = np.stack(suppliersizes)
Z = [arr[availability_random_index] for arr in Zs_avail]
Anew = suppliersizes.T*(mu_avail+P_avail@Z)
# Check that the sum of each column of Anew is within the bounds min and max defined in the input data Excel file
for i in range(n):
    column_sum = Anew[:,i].sum()
    if column_sum < feedstocks_a[i][2] or column_sum > feedstocks_a[i][1]:
        print(f'Warning: total availability of feedstock {i} is out of bounds!')

# Check constraint formulation for costs by taking one realization from Zs
item_presence = np.where(A != 0, 1, 0)
Z = [arr[cost_random_index] for arr in Zs_cost]
Cpnew = item_presence*(mu_cost+P_cost@Z)
# Check that each element different from 0 of Anew is within the bounds min and max defined in the input data Excel file
# If not, print a warning message
for i in range(n):
    for j in range(m):
        if Cpnew[j][i] != 0:
            if Cpnew[j][i] < feedstocks_costs[i][2] or Cpnew[j][i] > feedstocks_costs[i][1]:
                print(f'Warning: purchase cost of feedstock {i} from supplier {j} is out of bounds!')

# Compute maximum variation of BMP (toward direction of interest for robustification i.e. BMP that are lower than the mean expected values)
bmpdev = 1 - bmp_data[2]/bmp_data[0]
#print(bmpdev)

### Robust problem

In [ ]:
# SOLVE THE ROBUSTIFIED PROBLEM (with consequent HRT optimization)
# Which uncertainty is considered? You can turn on/off some uncertainty sources by changing the boolean variables below
uncertain_BMPs = False
uncertain_costs = True

# Find indices correspondent to a certain cluster to apply the 'regulating authority constraints' (RL vector in the math. formulation)
table1A = np.isin(itemtype,['S','B'])
table1B = np.isin(itemtype,['P'])
slurries = np.isin(itemtype,['S'])
secondharvest = np.isin(itemtype,['P2'])

#Extract value from 'bmp_curve' at the given HRT
HRT = 20 # USER MODIFY THIS VALUE TO RANGE FOR DIFFERENT HRT's: must be an integer between HRT_{min} and HRT_{max} defined in 'bnds' 
HRTminus1 = 1/HRT
BMP = []
for i in range(len(data['Item characteristics'].values[:,0])):
    BMPsubstrate = bmp_curve[i][HRT-1]
    BMP.append(BMPsubstrate)

# Create model
model=ro.Model('Biomethane supply-chain robust real')

# Define decision variables
s = model.dvar(1) # Lower-bound for revenues
c = model.dvar(1) # Upper-bound for costs
x = model.dvar((n,m)) # The amounts of each feedstock to be purchased from each supplier over the purchase horizon N (ton_{FM})
z = model.rvar(n) # Random variables = perturbation to availabilities
w = model.rvar(n) # Random variables = perturbation to costs
l = model.rvar(n) # Random variables = perturbation to BMP
# HYPOTHESIS: Budgeted sets for all the three uncertainties
availabilitySet = (-1<=z, z<=1, rso.norm(z,1)<=Gamma_availability) # Define 'availability uncertainty set'
costsSet = (-1<=w, w<=1, rso.norm(w,1)<=Gamma_cost) # Define 'costs uncertainty set'
BMPSet = (-1<=l, l<=1, rso.norm(l,1)<=Gamma_BMP) # Define 'BMP uncertainty set'

# Objective function (see "RobustOptimization_Project_Report.pdf" for details)
model.max(s-c)  # Maximize the net revenue over the purchase horizon N (€/day). Try to push s up and c down as much as possible to not violate the robust constraints below
if uncertain_BMPs == True:
  model.st((s <= Nminus1*(p_ch4*(VS*BMP*(1+bmpdev*l)*(1/1000))@x.sum(axis=1))).forall(BMPSet)) #uncomment to include BMP uncertainty
else:
  model.st(s <= Nminus1*(p_ch4*(VS*BMP*(1/1000))@x.sum(axis=1))) #comment to include BMP uncertainty
if uncertain_costs == True:
  model.st((c >= Nminus1*(((item_presence*(mu_cost+P_cost@w)).T*x).sum() + (Ct.T*x).sum())).forall(costsSet)) #uncomment to include price uncertainty
else:
  model.st(c >= Nminus1*((Cp.T*x).sum() + (Ct.T*x).sum())) #comment to include price uncertainty

# Constraints
model.st(x[table1A].sum() >= RL[0]*x.sum()) # Constraints from regulating authority
model.st(x[table1B].sum() <= RL[1]*x.sum()) # Constraints from regulating authority
model.st(x[secondharvest].sum() <= RL[2]*x.sum()) # Constraints from regulating authority
model.st(x[slurries].sum() >= RL[3]*x.sum()) # Constraints from regulating authority
model.st((x[:,0:-1] <= (suppliersizes[:,0:-1].T*(mu_avail+P_avail@z)).T).forall(availabilitySet)) # Availability constraint
for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st((((BMPinf-BMP)*(1+bmpdev*l))@(x.sum(axis=1)) <= (0.15*BMPinf*(1+bmpdev*l))@(x.sum(axis=1))).forall(BMPSet)) # Constraint to limit output residual BMP_{res} in digestate (environmental regulating authority limit)
model.st(Vr*N == HRT*x.sum()) # HRT hard constraint
model.st((x.T[-1] == A[-1])) # Constraint to consume feedstocks produced at 'myplant' i.e. named 'z' in the input data Excel example file
model.st(x >= 0) # Non-negativity constraint

# Solve the model
model.solve(my_solver)
opt_obj_robust_real = model.get() # Optimized value of the objective function
opt_x_robust_real = x.get() # Optimized decision variables
opt_s_real = s.get() # Optimized value of the lower-bound for revenues
opt_c_real = c.get() # Optimized value of the upper-bound for costs

# Print results
np.set_printoptions(suppress=True)
print('The net revenue for the purchase horizon of interest is', round(opt_obj_robust_real,2),
      'euro/day and the optimal fluxes of items to be purchased are', np.round(opt_x_robust_real, decimals=2))

In [ ]:
# Compute some quantities of interest with the solution obtained above
x = opt_x_robust_real # Optimal decision variables
s = opt_s_real # Optimized value of the lower-bound for revenues
c = opt_c_real # Optimized value of the upper-bound for costs
J = opt_obj_robust_real # Optimal objective function value
HRT = Vr*N/x.sum() # Check HRT value is correct
print(f'Checked HRT is {round(HRT,2)} days \n')
f_obj = s[0] # Biomethane gross revenue (€/day)
f_obj3 = (Ct.T*x).sum()/N # Transportation cost (€/day)
f_obj2 = c[0] - f_obj3 # Feedstocks' purchase cost (€/day)
print(f'Biomethane gross revenue is {round(f_obj,2)} €/day')
print(f'Feedstocks\' purchase cost is {round(f_obj2,2)} €/day')
print(f'Transportation cost is {round(f_obj3,2)} €/day')
print(f'Net revenue is {round(f_obj-f_obj2-f_obj3,2)} €/day \n')

# Optimal diet mix composition
diet = []
for i,name in zip(range(len(x)),data['Item characteristics'].values[:,0]):
    itempercentage = x[i].sum()/x.sum()*100
    print(f'{name} is {round(itempercentage,2)}% of the influent diet')
    diet.append(round(itempercentage,2))
print('')

# Save to Excel
results = [Gamma_availability, Gamma_cost, Gamma_BMP, HRT, f_obj, f_obj2, f_obj3, J, diet, uncertain_BMPs, uncertain_costs]
file_path = 'Results.xlsx'
sheet_name = 'R-LP_real'
save_list_to_excel(results, file_path, sheet_name)

# Check that the regulating authority and technical constraints have been respected
table1A = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S','B'])]).sum()
table1B = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P'])]).sum()
slurries = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['S'])]).sum()
secondharvest = np.array([s for f, s in zip(itemtype, x) if any(target in f for target in ['P2'])]).sum()
deltarl = [table1A*100/x.sum() - 70, 30 - table1B*100/x.sum(), slurries*100/x.sum() - 40, 20 - secondharvest*100/x.sum()]
for i in range(len(deltarl)):
    active = [deltarl[i] if deltarl[i]<0 else 0]
    print(f'Autorithy constraint {i} not respected by {round(active[0],2)}')

#
for i in range(np.size(k,1)):
    deltamax = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][1] # Technical constraint (max)
    deltamin = k[:,i]@(x.sum(axis=1))/x.sum() - bnds[i][0] # Technical constraint (min)
    print(f'Technical constriant {i} has margin to UB equal to {round(deltamax,2)}')
    print(f'Technical constriant {i} has margin to LB equal to {round(deltamin,2)}')
deltaHRTmax = Vr*N/x.sum() - bnds[7][1] # HRT constraint (max)
print(f'HRT constriant has margin to UB equal to {round(deltaHRTmax,2)}')
deltaHRTmin = Vr*N/x.sum() - bnds[7][0] # HRT constraint (min)
print(f'HRT constriant has margin to LB equal to {round(deltaHRTmin,2)}')

### Testing

In [ ]:
# TEST THE ROBUST SOLUTION AGAINST NEW RANDOM REALIZATIONS OF THE UNCERTAINTIES
# Generate some additional random realizations for testing
# Example: generate 20 new random realizations of feedstock purchase costs
years_test = 20 # Number of years you want to simulate the availability of data (i.e. remaining years of plant operation. Lifetime hypothesized 25 years))
np.random.seed(5) # Set random seed for reproducibility
feedstocks_costs_test= []
d=np.array([beta_sampler(barley) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(chicken) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(cow_d) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(cow_s) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(maize) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(pig) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(sorghum) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(tomato) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(triticale) for _ in range(years_test)])
feedstocks_costs_test.append(d)
d=np.array([beta_sampler(wheat) for _ in range(years_test)])
feedstocks_costs_test.append(d)
Cp_test_set = [item_presence*np.array(feedstocks_costs_test)[:,i] for i in range(years_test)]

In [ ]:
# Define function to compute objective function J given the x resulted from the problems solved above and the Cp testing realizations
def J_on_test(x,Cp):
    J = Nminus1*(p_ch4*(VS*BMP/1000)@x.sum(axis=1)-(Cp.T*x).sum()-(Ct.T*x).sum())
    return J
# At a given HRT (the one defined above): compute J values for each testing realization and for both robust and deterministic solutions
J_test_robust_real = [J_on_test(opt_x_robust_real,Cp_test_set[i]) for i in range(years_test)]
J_test_deterministic_real = [J_on_test(opt_x_real,Cp_test_set[i]) for i in range(years_test)]

# Plot the boxplot of testing J values for both robust and deterministic solutions
plt.boxplot(J_test_robust_real, vert=True, positions=[1], meanline=True, showmeans=True, meanprops={'color': 'r', 'linewidth': 2})
plt.boxplot(J_test_deterministic_real, positions=[2], vert=True, meanline=True, showmeans=True, meanprops={'color': 'r', 'linewidth': 2})
plt.scatter([1], opt_obj_robust_real, color='b', label='R-LP J value')  # Highlight other value
plt.scatter([2], opt_obj_real, color='green', label='D-LP J value')  # Highlight other value
plt.xticks([1, 2], ['R-LP', 'D-LP'])
plt.ylabel('Objective function J value (€ day$^{-1}$)')
plt.legend()
#plt.savefig(f'images/Testing_{HRT}_{current_date}.png')
plt.show()

### Plot

In [ ]:
# PLOT SOME INPUT DATA 
# Note: you must run the "Load and prepare the input data" section at the beginning of this script before running this part
# Plot net profit and BMP for different HRT values for each feedstock

fig, ax1 = plt.subplots(figsize=(8,6))

pos1 = ax1.get_position() # Get the position and size of the first subplot
ax2 = fig.add_axes([pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]) # Create a new axes object within the same figure

for i in range(n): # Range for all feedstocks
  # Plot net profit (Revenue - Purchase cost) curve 
  ax1.plot(p_ch4*VS[i]*bmp_curve[i]/1000-data['Purchase costs'].values[0,1:11].astype('float64')[i],label=data['Item characteristics'].values[:,0][i])

  # Plot BMP curve
  ax2.plot(bmp_curve[i],label=data['Item characteristics'].values[:,0][i])

ax1.set_xlim(0,49)
ax2.set_ylim(0,350)
ax2.set_xlim(0,49)
ax1.set_xlabel('HRT (d)')
ax1.set_ylabel('Net profit (€ ton$_{FM}^{-1}$)')
ax2.set_xlabel('HRT (d)')
ax2.set_ylabel('BMP (Nm$^3_{ch4}$ ton$_{FM}^{-1}$)')
ax2.legend(bbox_to_anchor=(1, 1), loc='upper left')
ax1.grid()
ax2.grid()

plt.savefig(f'images/NetProfitBMP_{current_date}.png', bbox_inches='tight')
# plt.show()

In [ ]:
# PLOT SOME DATA: STANDARDIZED COST DATA REALIZATIONS TO CHECK THE SHAPE OF THE DISTRIBUTIONS
# Note: you must run up to the "Design of uncertainty sets" section before running this part
# Plot distributions of cost data (standardized to Zs) for each feedstock

fig, ax = plt.subplots(1, 1)

for i in np.arange(n):
    # Plot histogram of standardized cost data realizations
    ax.hist(Zs_cost[i,:], density=True, bins='auto', histtype='stepfilled', alpha=0.2, label = data['Item characteristics'].values[:,0][i])

ax.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'images/Distribution_cost_{current_date}.png', bbox_inches='tight')
# plt.show()

In [ ]:
# PLOT THE RESULTS COLLECTED IN THE EXCEL FILE
# Plot deterministic solutions for different HRT values

datatoplot = pd.read_excel('Results.xlsx', sheet_name='D-LP_real (2)') # Read the Excel file and select the desired sheet
filtered_df = datatoplot.iloc[:,:] # Filter rows
x_values = filtered_df['HRT']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k]) # Sort the data based on x_values
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]
bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1) # Plot the stacked bar chart
ax1.set_ylim(0,100)
ax1.set_xlabel('HRT (days)')
rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

# Get the position and size of the first subplot
pos1 = ax1.get_position() # Get the position and size of the first subplot
ax2 = fig.add_axes([pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]) # Create a new axes object within the same figure
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black') # Plot the line chart
ax2.set_xlabel('HRT (days)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')
plt.tight_layout()
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'images/Screening_HRT_D-LP_{current_date}.png', bbox_inches='tight')
# plt.show()

In [ ]:
# Plot robust solutions for different HRT values

datatoplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (2)') # Read the Excel file and select the desired sheet
filtered_df = datatoplot.iloc[6:,:] # Filter rows
x_values = filtered_df['HRT']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column
bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

other_data.plot(kind='bar', stacked=True, ax=ax1) # Plot the stacked bar chart
ax1.set_ylim(0,100)
ax1.set_xlabel('HRT (days)')
ax1.set_xticklabels(round(x_values,2))

pos1 = ax1.get_position() # Get the position and size of the first subplot
ax2 = fig.add_axes([pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]) # Create a new axes object within the same figure
ax2.plot(x_values, y_values, marker='o', color='black') # Plot the line chart
ax2.set_xlabel('HRT (days)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')
plt.tight_layout()
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'images/Screening_HRT_R-LP_{current_date}.png', bbox_inches='tight')
# plt.show()

In [ ]:
# Plot robust solutions for different Prob values at a given HRT value (HRT=20 days in this cell)

datatoplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (3)') # Read the Excel file and select the desired sheet

filtered_df = datatoplot.iloc[0:13,:] # Filter rows
x_values = filtered_df['Prob']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k]) # Sort the data based on x_values
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]
bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1) # Plot the stacked bar chart
ax1.set_ylim(0,100)
ax1.set_xlabel('Prob (-)')
rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

pos1 = ax1.get_position() # Get the position and size of the first subplot
ax2 = fig.add_axes([pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]) # Create a new axes object within the same figure
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black') # Plot the line chart
ax2.set_xlabel('Prob (-)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')
plt.tight_layout()
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'images/Screening_Prob_R-LP_20_{current_date}.png', bbox_inches='tight')
# plt.show()

In [ ]:
# Plot robust solutions for different Prob values at a given HRT value (HRT=50 days in this cell)

datatoplot = pd.read_excel('Results.xlsx', sheet_name='R-LP_real (3)') # Read the Excel file and select the desired sheet

filtered_df = datatoplot.iloc[13:,:] # Filter rows
x_values = filtered_df['Prob']  # Select values from the first row, excluding the first column
y_values = filtered_df['J'] # Select values from the second row, excluding the first column
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values.values[k]) # Sort the data based on x_values
x_values_sorted = [x_values.values[i] for i in sorted_indices]
y_values_sorted = [y_values.values[i] for i in sorted_indices]
bar_columns = filtered_df.columns[8:18]
other_data = filtered_df[filtered_df.columns[8:18]]

# Create subplots with a single row and one column
fig, ax1 = plt.subplots()

other_data= other_data.iloc[sorted_indices]
other_data.plot(kind='bar', stacked=True, ax=ax1) # Plot the stacked bar chart
ax1.set_ylim(0,100)
ax1.set_xlabel('Prob (-)')
rounded_values = list(map(lambda x: round(x, 2), x_values_sorted))
ax1.set_xticklabels(rounded_values)

pos1 = ax1.get_position() # Get the position and size of the first subplot
ax2 = fig.add_axes([pos1.x0 + 1.3, pos1.y0, pos1.width, pos1.height]) # Create a new axes object within the same figure
ax2.plot(x_values_sorted, y_values_sorted, marker='o', color='black') # Plot the line chart
ax2.set_xlabel('Prob (-)')
ax2.set_ylabel('Net profit (€ day$^{-1}$)')
plt.tight_layout()
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left')
plt.savefig(f'images/Screening_Prob_R-LP_50_{current_date}.png', bbox_inches='tight')
# plt.show()